# SR/VR one-file data check

Run one skimmed DoubleMuon data ROOT file through the topology-specific 2mu2e/4mu SR and VR test channels. LJ isolation and cosmic-veto cuts are intentionally omitted for the ABCD scan.

In [ ]:
import sys
import os
import importlib

from coffea import processor
import coffea.util
import matplotlib.pyplot as plt
import pandas as pd

sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path:
    sys.path.insert(1, sidm_path)

from sidm.tools import llpnanoaodschema, scaleout, sidm_processor, utilities

importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)

utilities.set_plot_style()
%matplotlib inline


In [ ]:
runner = processor.Runner(
    executor=processor.IterativeExecutor(),
    schema=llpnanoaodschema.LLPNanoAODSchema,
    skipbadfiles=False,
)

channels = [
    "test_SR_2mu2e",
    "test_VR_2mu2e",
    "test_SR_4mu",
    "test_VR_4mu",
]
iso_2d_histogram_names = ["mulj_egmlj_iso", "mulj_mulj_iso"]
cut_1d_histogram_names = [
    "abcd_2mu2e_invmass",
    "abcd_2mu2e_absdphi",
    "abcd_4mu_invmass",
    "abcd_4mu_absdphi",
]
iso_3d_histogram_names = [
    "abcd_2mu2e_iso_invmass",
    "abcd_2mu2e_iso_absdphi",
    "abcd_4mu_iso_invmass",
    "abcd_4mu_iso_absdphi",
]
histogram_names = iso_2d_histogram_names + cut_1d_histogram_names + iso_3d_histogram_names

p = sidm_processor.SidmProcessor(
    channels,
    ["abcd_iso_base"],
    unweighted_hist=False,
)


In [ ]:
sample = "DoubleMuon_2018C"
fileset = utilities.make_fileset(
    [sample],
    "llpNanoAOD_v2",
    max_files=10,
    location_cfg="data_skimmed.yaml",
)

for sample, sample_fileset in fileset.items():
    print(sample)
    for root_file in sample_fileset["files"]:
        print(f"  {root_file}")


In [ ]:
output_data = runner.run(
    fileset,
    treename="Events",
    processor_instance=p,
)

In [ ]:
rows = []
sample_output = output_data["out"][sample]
sample_hists = sample_output["hists"]
assert set(channels).issubset(set(sample_hists[histogram_names[0]].axes[0])), "Missing channel axis entries"

for channel in channels:
    final_cut = list(sample_output["cutflow"][channel].rows.values())[-1]
    for histogram_name in histogram_names:
        histogram = sample_hists[histogram_name][{"channel": channel}]
        rows.append({
            "sample": sample,
            "channel": channel,
            "final_raw_events": int(final_cut["raw"]),
            "histogram": histogram_name,
            "yield": float(histogram.values().sum()),
        })

summary = pd.DataFrame(rows)
summary

In [ ]:
for channel in channels:
    topology = "2mu2e" if channel.endswith("2mu2e") else "4mu"
    relevant_1d_histograms = [name for name in cut_1d_histogram_names if topology in name]
    relevant_2d_histograms = ["mulj_egmlj_iso"] if topology == "2mu2e" else ["mulj_mulj_iso"]
    relevant_3d_histograms = [name for name in iso_3d_histogram_names if topology in name]

    for histogram_name in relevant_1d_histograms:
        histogram = sample_hists[histogram_name][{"channel": channel}]
        utilities.plot(histogram, flow="none")
        plt.title(f"{sample} - {channel} - {histogram_name}")
        plt.show()
        plt.close()

    for histogram_name in relevant_2d_histograms:
        histogram = sample_hists[histogram_name][{"channel": channel}]
        utilities.plot(histogram, flow="none")
        plt.title(f"{sample} - {channel} - {histogram_name}")
        plt.show()
        plt.close()

    for histogram_name in relevant_3d_histograms:
        histogram = sample_hists[histogram_name][{"channel": channel}]
        iso_axis_names = [axis.name for axis in histogram.axes[:2]]
        kinematic_axis_name = histogram.axes[2].name

        iso_plane = histogram.project(*iso_axis_names)
        utilities.plot(iso_plane, flow="none")
        plt.title(f"{sample} - {channel} - {histogram_name} (kinematic integrated)")
        plt.show()
        plt.close()

        kinematic_projection = histogram.project(kinematic_axis_name)
        utilities.plot(kinematic_projection, flow="none")
        plt.title(f"{sample} - {channel} - {histogram_name} (isolation integrated)")
        plt.show()
        plt.close()
